## LOAD PDF

In [1]:
from langchain_pymupdf4llm import PyMuPDF4LLMLoader

file_path = "insurance.pdf"
pypdf_loader = PyMuPDF4LLMLoader(file_path)
pdf_docs = pypdf_loader.load()

/Users/ravijot/Desktop/code/Langchain_Langgraph_1.3.2/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print("Length of PDF Docs : ", len(pdf_docs))

Length of PDF Docs :  36


In [3]:
print(pdf_docs[1].page_content)

## POLICY HOLDER’S
# INSURANCE GUIDE

Version: GBKPH.ver.02-09-19 BKL ENG


**CONTENTS**
**Introduction..................................................1**
**Insurance Policy Life cycle..........................1**


**Understanding the policy document..........4**
**1.** **Information Page**
**2.** **First Premium Receipt**
**3.** **Key Feature document**
**4.** **Policy Schedule**
**5.** **Policy booklet**
**6.** **Index**
**7.** **Annexure**
**8.** **Ombudsman Centre details**
**9.** **Key Personal Information**
**10.** **Scanned copy of proposal form**


**Connecting with the insurer......................7**
**1.** **Nearest SBI Life Branch**
**2.** **Call Centre**
**3.** **Email**
**4.** **SMS**
**5.** **Write to us**
**6.** **Website**
**7.** **Customer Self Service Portal**
**8.** **Mobile Application**


**Renewing the Policy...................................12**
**1.** **Through Website**
**2.** **EFT**
**3.** **Direct Remittance**
**4.** **Net Banking**
**5.** **SI-Credit C

In [4]:
print(pdf_docs[1].metadata)

{'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 14.0 (Macintosh)', 'creationdate': '2019-10-25T10:22:06+05:30', 'source': 'insurance.pdf', 'file_path': 'insurance.pdf', 'total_pages': 36, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2019-10-25T10:22:08+05:30', 'trapped': '', 'modDate': "D:20191025102208+05'30'", 'creationDate': "D:20191025102206+05'30'", 'page': 1}


In [5]:
from pathlib import Path

for doc in pdf_docs:
    doc.metadata.update({
        "source": Path(file_path).name,
        "document_type": "insurance_policy"
    })

In [6]:
print(pdf_docs[1].metadata)
print(pdf_docs[1].id)

{'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 14.0 (Macintosh)', 'creationdate': '2019-10-25T10:22:06+05:30', 'source': 'insurance.pdf', 'file_path': 'insurance.pdf', 'total_pages': 36, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2019-10-25T10:22:08+05:30', 'trapped': '', 'modDate': "D:20191025102208+05'30'", 'creationDate': "D:20191025102206+05'30'", 'page': 1, 'document_type': 'insurance_policy'}
None


## MERGE DOC BEFORE CHUNKING

In [110]:
full_text = ""

for doc in pdf_docs:
    page = doc.metadata.get("page")

    full_text += f"\n\n=== PAGE {page} ===\n\n"
    full_text += doc.page_content

## CHUNKING PDF 

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=300,
    separators=[
        "\n# ",
        "\n## ",
        "\n### ",
        "\n\n",
        "\n",
        " ",
        ""
    ]
)

# chunks = splitter.split_documents(pdf_docs)
chunks = splitter.split_text(full_text)


print(f"Total chunks: {len(chunks)}")

Total chunks: 38


In [107]:
print(chunks[0].id)

None


## EMBEDDINGS

In [108]:
from langchain_openai import OpenAIEmbeddings
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

#Load environment variables from .env file
load_dotenv()

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-large",
    # With the `text-embedding-3` class
    # of models, you can specify the size
    # of the embeddings you want returned.
    # dimensions=1024
)

## VECTOR STORE

In [109]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="insurance_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db",  # Where to save data locally, remove if not necessary
)

## ADD DOCUMENTS

In [97]:
vector_store.add_documents(documents=chunks)

['f8d40eb9-813c-470f-b40f-b14e3c6b0afd',
 'd2092065-5d9a-4ddf-80dd-d8e6843c6655',
 'a6421778-a676-483e-ae5d-7fd23916ed93',
 'd09932c6-ed71-421e-92af-a0729bcb895f',
 '1da909a6-61dc-4d9d-b858-bf32fee0bce0',
 'b84d6135-f961-4fe5-b57c-b3020fa4901f',
 '9cdf0812-ee55-4e47-ad14-dde768beee2b',
 '40494c81-4654-4368-8b3c-3d6779a16d3a',
 '636e82ab-ede4-4a40-9a20-ed1fc1bf16a2',
 '3ef942ea-e58f-4728-8b65-a0974614ac60',
 'b8e138f3-ba57-4578-8875-0e6b143513e7',
 '912fdc0a-7e9f-4fbd-85c1-6e55db7b9471',
 'f0df6d05-c4ec-47fa-9c5d-aaa6e366625e',
 'ae040b4c-ebb6-474a-b38e-d73d1384e10e',
 '8cb0620b-090b-4eba-b116-ef4121f6baf5',
 'e3cb67de-2df4-4654-8183-d657582ac75b',
 '9231af6c-f30e-4a24-8a00-48d1a149d8fd',
 '4931e28b-7eb9-4b50-80c4-7e4ea03ae883',
 'f80943f0-f709-42e0-8ece-501f5f658a8c',
 'c1174c59-057c-4748-a257-9d1e2c4b0524',
 '08f012b6-ea35-494f-ae6e-469328d27b36',
 'f37ab022-4a07-476c-aa83-c454e336f32d',
 '2f635034-57a5-417d-a9b5-d17886bf7592',
 '63912faf-2336-4b73-bfde-2f6d0e096d41',
 'daa8abfb-5f8e-

In [27]:
similar_docs = vector_store.similarity_search("How can I connect to the insurer?", k=5)

In [42]:
print(similar_docs[0].__dict__)

{'id': 'ed666279-6cde-41af-b1bd-843560470473', 'metadata': {'creationdate': '2019-10-25T10:22:06+05:30', 'trapped': '', 'file_path': 'insurance.pdf', 'title': '', 'producer': 'Adobe PDF Library 15.0', 'format': 'PDF 1.4', 'source': 'insurance.pdf', 'subject': '', 'author': '', 'document_type': 'insurance_policy', 'moddate': '2019-10-25T10:22:08+05:30', 'creationDate': "D:20191025102206+05'30'", 'total_pages': 36, 'keywords': '', 'creator': 'Adobe InDesign 14.0 (Macintosh)', 'page': 10, 'modDate': "D:20191025102208+05'30'"}, 'page_content': 'Policy Holder’s Insurance Guide | 2019\n\n\n**Connecting with the insurer**\n\n\nThe policy holder can connect with the insurer through various\nmodes.\n\n\n**1. Visit the nearest SBI Life branch:**\n\n\nThe nearest service branch identified on the basis of the\ncontact details shared by the policy holder is given in the policy\ndocument on the Welcome letter. The branch locator on the\ncompany website (www.sbilife.co.in) can be used to locate\nothe

In [40]:
similar_docs_with_scores = vector_store.similarity_search_with_score("How can I connect to the insurer?", k=5)

In [52]:
# print(similar_docs_with_scores[0])
# print(similar_docs_with_scores[0][1])
for doc in similar_docs_with_scores:
    print(doc[1])

0.8627855181694031
1.1158077716827393
1.1436548233032227
1.148887038230896
1.1499674320220947


In [48]:
similar_docs_with_relevance_scores = vector_store.similarity_search_with_relevance_scores("How can I connect to the insurer?", k=5)

In [53]:
# print(similar_docs_with_relevance_scores[0][1])
for doc in similar_docs_with_scores:
    print(doc[1])

0.8627855181694031
1.1158077716827393
1.1436548233032227
1.148887038230896
1.1499674320220947


In [61]:
retriever = vector_store.as_retriever(search_type="mmr", search_kwargs={"k":5, "fetch_k":12})
similar_docs_with_retriever = retriever.invoke("How can I connect to the insurer?")

In [70]:
print(similar_docs_with_retriever[0].__dict__)


{'id': 'ed666279-6cde-41af-b1bd-843560470473', 'metadata': {'modDate': "D:20191025102208+05'30'", 'producer': 'Adobe PDF Library 15.0', 'total_pages': 36, 'trapped': '', 'author': '', 'document_type': 'insurance_policy', 'title': '', 'page': 10, 'moddate': '2019-10-25T10:22:08+05:30', 'source': 'insurance.pdf', 'creationDate': "D:20191025102206+05'30'", 'file_path': 'insurance.pdf', 'creationdate': '2019-10-25T10:22:06+05:30', 'format': 'PDF 1.4', 'subject': '', 'keywords': '', 'creator': 'Adobe InDesign 14.0 (Macintosh)'}, 'page_content': 'Policy Holder’s Insurance Guide | 2019\n\n\n**Connecting with the insurer**\n\n\nThe policy holder can connect with the insurer through various\nmodes.\n\n\n**1. Visit the nearest SBI Life branch:**\n\n\nThe nearest service branch identified on the basis of the\ncontact details shared by the policy holder is given in the policy\ndocument on the Welcome letter. The branch locator on the\ncompany website (www.sbilife.co.in) can be used to locate\nothe

In [76]:
context = "\n\n".join(
    f"Page: {doc.metadata.get('page')}\n{doc.page_content}"
    for doc in similar_docs
)

In [77]:
print(context)

Page: 10
Policy Holder’s Insurance Guide | 2019


**Connecting with the insurer**


The policy holder can connect with the insurer through various
modes.


**1. Visit the nearest SBI Life branch:**


The nearest service branch identified on the basis of the
contact details shared by the policy holder is given in the policy
document on the Welcome letter. The branch locator on the
company website (www.sbilife.co.in) can be used to locate
other branches as well. A similar branch locator is available in
the mobile application “Easy Access” which can be downloaded
from Play store/appstore.


**2. Make a call to the Call center**


Policy holders can call the toll free number 1800-267-9090 any
time between 9.00 am to 9.00 pm 7 days a week.


**3. E-Mail to us**


Policy holders can mail their queries and complaints to the
generic email id info@sbilife.co.in.


**4. Send an SMS**

Page: 12
Policy Holder’s Insurance Guide | 2019


forms perform limited policy related services and gain
informa

## RAG

In [102]:
from langchain_core.prompts import PromptTemplate
from langchain.chat_models import init_chat_model

model = init_chat_model(model="gpt-5.4")

prompt = PromptTemplate.from_template("""
You are a helpful SBI insurance assistant.

Answer ONLY using the provided context.

If the context contains a procedure,
instructions, numbered list, or steps:

- Include ALL steps.
- Do not summarize.
- Do not omit any step.
- Preserve the order of the steps.

If the answer is not present in the context,
say "I could not find that information in the policy document."

Question:
{query}

Context:
{context}
""")

chain = prompt | model 

# query = "How do I contact the insurer?"
query = "How can I renew my policy?"

# similar_docs = vector_store.similarity_search(query, k=5)
retriever = vector_store.as_retriever(search_type="mmr", search_kwargs={"k":10, "fetch_k":30})
similar_docs_with_retriever = retriever.invoke(query)

context = "\n\n".join(
    f"Page: {doc.metadata.get('page')}\n{doc.page_content}"
    for doc in similar_docs_with_retriever
)

print(chain.invoke({"query":query, "context": context}).content)

You can renew your policy when the premium due for the policy has been received by the insurer within the grace period. The policy then gets renewed and the status is updated as active.

Various options available with the policy holder to make the renewal premium payment are as given below:

**1. SBI Life Website**  
Policy holders can make premium payment directly from the website in the Pay Premium Online option under Services section. Alternatively, one can also make payments by login to the customer self service portal available on the landing page of the website.

**2. Electronic Fund Transfer (EFT)**  
Through this option, by paying premium either in cash (upto Rs 49,999) or through cheque or direct debit, the premium will be sent through EFT from the SBG branch where the premium is received.

**3. Direct Remittance**  
Policy holder may approach the nearest SBI Life branch and pay the premium through cheque/demand draft.

**4. Net Banking**  
Policy holder can pay premium throug

In [103]:
print(context)

Page: 15
**Renewing your policy**


The policy gets renewed and the status is updated as active
when the premium due for the policy has been received by the
insurer within the grace period.
Various options available with the policy holder to make the
renewal premium payment are as given below.


**1. SBI Life Website**


Policy holders can make premium payment directly from the
website in the Pay Premium Online option under Services
section. Alternatively, one can also make payments by login to
the customer self service portal available on the landing page
of the website.


**2. Electronic Fund Transfer (EFT)**


Through this option, by paying premium either in cash (upto
Rs 49,999) or through cheque or direct debit, the premium will
be sent through EFT from the SBG branch where the premium
is received.


**3. Direct Remittance**


Policy holder may approach the nearest SBI Life branch and
pay the premium through cheque/demand draft.


**4. Net Banking**
Policy holder can pay premium t